In [1]:
from langchain.document_loaders import PyPDFLoader, TextLoader, DirectoryLoader

# Load a single PDF
pdf_loader = PyPDFLoader("data/attention.pdf")
pdf_docs = pdf_loader.load()

# Load multiple text files from a folder
folder_loader = DirectoryLoader("data/", glob="**/*.txt")
text_docs = folder_loader.load()

print(f"PDF docs: {len(pdf_docs)}, Text docs: {len(text_docs)}")

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


PDF docs: 15, Text docs: 1


In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Example: Splitting a PDF document into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # max tokens per chunk
    chunk_overlap=200     # overlap to maintain context
)

chunks = splitter.split_documents(pdf_docs)
print(f"Total chunks created: {len(chunks)}")
print(chunks[0].page_content[:300])  # preview first chunk

Total chunks created: 52
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [3]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Step 1: Load text file
loader = TextLoader("data/speech.txt")
docs = loader.load()

# Step 2: Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=10)
chunks = splitter.split_documents(docs)

# Each chunk now ready for embedding
print(chunks, "\n", chunks[0].metadata, "\n", chunks[0].page_content[:200])

[Document(metadata={'source': 'data/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the'), Document(metadata={'source': 'data/speech.txt'}, page_content='upon the tested foundations of political liberty. We have no selfish ends to'), Document(metadata={'source': 'data/speech.txt'}, page_content='ends to serve. We desire no conquest, no dominion. We seek no indemnities for'), Document(metadata={'source': 'data/speech.txt'}, page_content='for ourselves, no material compensation for the sacrifices we shall freely'), Document(metadata={'source': 'data/speech.txt'}, page_content='freely make. We are but one of the champions of the rights of mankind. We shall'), Document(metadata={'source': 'data/speech.txt'}, page_content='We shall be satisfied when those rights have been made as secure as the faith'), Document(metadata={'source': 'data/speech.txt'}, page_content='the faith and the freedom of nations can make them.'), Document(metadata={

In [5]:
import api_keys
from openai import OpenAI

# Initialize client with your API key from api_keys.py
client = OpenAI(api_key=api_keys.open_ai_key)

text = "Artificial Intelligence helps automate tasks."

# Create embeddings
response = client.embeddings.create(
    input=text,
    model="text-embedding-3-small"
)

vector = response.data[0].embedding
print(f"Vector length: {len(vector)}")

Vector length: 1536


In [13]:
import api_keys
from huggingface_hub import InferenceClient
import numpy as np

# Initialize the Hugging Face client
client = InferenceClient(api_key=api_keys.hugging_face)

# Define model and input sentences
model_id = "sentence-transformers/all-MiniLM-L6-v2"
sentences = ["AI automates tasks", "AI helps in automation"]

# ✅ Correct way: pass text directly, not as "inputs="
response = client.feature_extraction(model=model_id, text=sentences)

# Convert response to numpy array
embeddings = np.array(response)

print(embeddings.shape)  # Should print (2, 384)

(2, 384)


In [ ]:
import importlib
import api_keys
importlib.reload(api_keys)

# In case the env file is updated and new variables are added
# To avoid Python reading from cache

In [1]:
import ollama
import numpy as np

# Define your model and input sentences
model_name = "mxbai-embed-large"
sentences = ["AI automates tasks", "AI helps in automation"]

# Get embeddings for multiple sentences
embeddings = []

for text in sentences:
    response = ollama.embeddings(model=model_name, prompt=text)
    embeddings.append(response["embedding"])

# Convert to numpy array
embeddings = np.array(embeddings)

print(embeddings.shape)  # e.g., (2, 1024)

(2, 1024)


In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

v1 = np.array([0.2, 0.3, 0.5])
v2 = np.array([0.1, 0.9, 0.5])

score = cosine_similarity([v1], [v2])[0][0]
print(f"Similarity Score: {score:.3f}")

Similarity Score: 0.847


In [5]:
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.schema import Document
import api_keys  # your dotenv loader

# Initialize embeddings
embeddings = OpenAIEmbeddings(api_key=api_keys.open_ai_key)

# Example document chunks
chunks = [
    Document(page_content="Data privacy ensures protection of user information."),
    Document(page_content="Data security focuses on safeguarding digital assets."),
    Document(page_content="AI and data privacy intersect when handling sensitive information."),
]

# Create a FAISS vector database
vector_db = FAISS.from_documents(chunks, embeddings)

# Save for later use
vector_db.save_local("faiss_index")

# Reload and query (allow dangerous deserialization needed for FAISS)
db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

# Run similarity search
results = db.similarity_search("What is data privacy?", k=2)

# Display results
for i, res in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(res.page_content[:200])


Result 1:
Data privacy ensures protection of user information.

Result 2:
AI and data privacy intersect when handling sensitive information.


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document
import torch

# STEP 2: Force CPU mode (important for AMD / low RAM)
torch.set_default_device("cpu")
torch.set_num_threads(2)
torch.set_num_interop_threads(2)

# STEP 3: Choose a lightweight embedding model (≈80 MB)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L3-v2"  # lighter than all-MiniLM-L6-v2
)

# STEP 4: Create example documents
chunks = [
    Document(page_content="Data security policies protect data from unauthorized access."),
    Document(page_content="Data privacy defines how information is collected and managed."),
    Document(page_content="Encryption ensures safe communication between systems."),
]

# STEP 5: Create or load Chroma vector store (persistent)
chroma_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_light_store"
)

# STEP 6: Save to disk
chroma_db.persist()
print("✅ Chroma lightweight DB persisted successfully!")

# STEP 7: Query
query = "Explain data security policy"
results = chroma_db.similarity_search(query, k=2)

for i, r in enumerate(results, 1):
    print(f"\n📄 Result {i}: {r.page_content[:150]}")

C:\Users\Utkarsh Mishra\AppData\Local\Temp\ipykernel_18812\1104384494.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
